In [173]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest, RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, BaggingRegressor
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel, SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor


In [174]:
df = pd.read_csv('owid-energy-data.csv')
print(df.shape)
print(df.isnull().mean()*100)

(23232, 130)
country                    0.000000
year                       0.000000
iso_code                  26.248278
population                19.249311
gdp                       49.294077
                            ...    
wind_elec_per_capita      64.544594
wind_electricity          59.848485
wind_energy_per_capita    77.853822
wind_share_elec           67.790117
wind_share_energy         72.542183
Length: 130, dtype: float64


In [175]:
# Keep only country-level data (iso_code is not null) and drop regional aggregates(like global , continental)
df = df[df['iso_code'].notna()]
target = 'renewables_share_energy'
df = df.dropna(subset=[target])#removing old data when no track of target var
print(df.shape)

(4459, 130)


In [176]:

manual_leakage = [col for col in df.columns if any(word in col for word in 
                 ['share', 'renewables', 'low_carbon', 'fossil', 'nuclear', 'hydro', 'solar', 'wind', 'biofuel'])]

manual_leakage = [c for c in manual_leakage if c != target]

# 2. Statistical filter for EVERYTHING ELSE
# Drop columns that are perfectly correlated with the target
corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
stat_leakage = corr_matrix.index[corr_matrix[target] > 0.8].tolist()
stat_leakage = [c for c in stat_leakage if c != target]

# 3. Combine and Drop
final_drop = list(set(manual_leakage + stat_leakage))
df = df.drop(columns=final_drop)

print(df.shape)

(4459, 50)


In [177]:
threshold = 0.6
df = df.dropna(thresh=int((1-threshold)*len(df)), axis=1)
print(df.shape)

(4459, 50)


In [178]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])


In [179]:
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])
df['energy_per_gdp'] = df['energy_per_capita'] / (df['gdp'] / df['population'] + 1)

In [180]:
iso = IsolationForest(contamination=0.03, random_state=42)
outliers = iso.fit_predict(df[num_cols])
df = df[outliers == 1]

In [181]:
df.to_csv('cleaned_owid_energy_data1.csv', index=False)

print(f"Cleaned dataset saved successfully! Total records: {len(df)}")

Cleaned dataset saved successfully! Total records: 4325


In [182]:
df.shape

(4325, 51)

In [183]:
X = df.drop(columns=[target, 'country', 'iso_code'])
y = df[target]

In [184]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=X.columns)

In [185]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

In [186]:
selections = {}

In [187]:
# Filter-Based (ANOVA/Correlation/Variance)
skb = SelectKBest(f_regression, k=10)
selections['Filter-Based'] = (skb.fit_transform(X_train, y_train), skb.transform(X_test))
print(selections['Filter-Based'][0].shape, selections['Filter-Based'][1].shape)

(3460, 10) (865, 10)


In [188]:
#Wrapper based
sfs = SequentialFeatureSelector(LinearRegression(), n_features_to_select=10, direction='forward')
selections['Wrapper-Based'] = (sfs.fit_transform(X_train, y_train), sfs.transform(X_test))
print(selections['Wrapper-Based'][0].shape, selections['Wrapper-Based'][1].shape)


(3460, 10) (865, 10)


In [189]:
# Embedded-Based (Lasso L1 Regularization)
lasso_sel = SelectFromModel(LassoCV(), threshold='median')
selections['Embedded-Based'] = (lasso_sel.fit_transform(X_train, y_train), lasso_sel.transform(X_test))
print(selections['Embedded-Based'][0].shape, selections['Embedded-Based'][1].shape)

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.682e+02, tolerance: 5.433e+01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.086e+02, tolerance: 5.433e+01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sc

(3460, 24) (865, 24)


In [190]:
# Feature Extraction (PCA)
pca = PCA(n_components=10)
selections['PCA-Extraction'] = (pca.fit_transform(X_train), pca.transform(X_test))
print(selections['PCA-Extraction'][0].shape, selections['PCA-Extraction'][1].shape)

(3460, 10) (865, 10)


In [191]:
selections['All-Features'] = (X_train, X_test)

In [192]:
models = {
    'LinearReg': LinearRegression(),
    'DecTree': DecisionTreeRegressor(max_depth=10),
    'RandForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'GradBoost': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=10),
    'Voting': VotingRegressor([('lr', LinearRegression()), ('rf', RandomForestRegressor(n_estimators=50)),('dt', DecisionTreeRegressor(max_depth=5))])
}

In [193]:
import time
import joblib

results = []
for sel_name, (xtr, xte) in selections.items():
    for mod_name, model in models.items():
        start_time = time.time()
        
        model.fit(xtr, y_train)
        
        end_time = time.time()
        training_time = round(end_time - start_time, 4)
        
        y_pred = model.predict(xte)
        joblib.dump(model,sel_name+"_"+mod_name+".joblib")
        results.append({
            'Selection': sel_name,
            'Model': mod_name,
            'R2': round(r2_score(y_test, y_pred), 4),
            'MAE': round(mean_absolute_error(y_test, y_pred), 4),
            'Train_Time_Sec': training_time  
        })
        
#output
comparison_df = pd.DataFrame(results).sort_values(by='R2', ascending=False)
print(comparison_df.head(15))

         Selection       Model      R2     MAE  Train_Time_Sec
26    All-Features  RandForest  0.9660  1.2710          6.6932
28    All-Features     Bagging  0.9620  1.3758          0.6923
14  Embedded-Based  RandForest  0.9604  1.4238          3.3661
8    Wrapper-Based  RandForest  0.9598  1.5846          1.5397
16  Embedded-Based     Bagging  0.9482  1.5971          0.3289
10   Wrapper-Based     Bagging  0.9413  1.8515          0.1584
27    All-Features   GradBoost  0.9092  2.9681          2.6480
13  Embedded-Based     DecTree  0.8824  2.8402          0.0373
25    All-Features     DecTree  0.8758  2.4450          0.0731
29    All-Features      Voting  0.8739  3.4630          3.1472
2     Filter-Based  RandForest  0.8709  2.3455          1.2908
15  Embedded-Based   GradBoost  0.8697  3.4689          1.3818
17  Embedded-Based      Voting  0.8637  3.5808          1.7874
4     Filter-Based     Bagging  0.8596  2.5213          0.1247
11   Wrapper-Based      Voting  0.8392  4.0950         

In [194]:
comparison_df.to_csv('model_performance_results_new.csv', index=False)